<a href="https://colab.research.google.com/github/tejashwinirk/Agentic-AI/blob/main/Copy_of_Lab8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langgraph langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.8 MB/s eta 0:00:00


In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key="ENTER_API_KEY",
    model="llama-3.1-8b-instant",
    temperature=0
)

print("Groq model connected successfully!")

Groq model connected successfully!


In [3]:
from typing import TypedDict

class TeamState(TypedDict):
    task: str
    worker_result: str
    summary: str

print("TeamState created successfully!")

TeamState created successfully!


In [4]:
def worker(state: TeamState) -> dict:
    answer = llm.invoke(
        "Solve this math problem, show the number only: "
        + state["task"]
    ).content

    return {
        "worker_result": answer
    }


def supervisor(state: TeamState) -> dict:
    summary = llm.invoke(
        f"The worker solved '{state['task']}' "
        f"and got {state['worker_result']}. "
        "Write a one-line summary."
    ).content

    return {
        "summary": summary
    }

print("Worker and Supervisor nodes created!")

Worker and Supervisor nodes created!


In [5]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(TeamState)

builder.add_node("worker", worker)
builder.add_node("supervisor", supervisor)

builder.add_edge(START, "worker")
builder.add_edge("worker", "supervisor")
builder.add_edge("supervisor", END)

graph = builder.compile()

print("LangGraph workflow created successfully!")

LangGraph workflow created successfully!


In [6]:
result = graph.invoke({
    "task": "What is 144 divided by 12, then plus 5?"
})

print("Worker result:", result["worker_result"])
print("Supervisor summary:", result["summary"])

Worker result: 12
Supervisor summary: The worker incorrectly solved the math problem 'What is 144 divided by 12, then plus 5?' and arrived at an answer of 12, which is actually the result of 144 divided by 12, not the final calculation.
